In [1]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from pyriemann.estimation import Covariances
from pyriemann.utils.mean import mean_riemann
from pyriemann.utils.distance import distance_riemann
from sklearn.model_selection import KFold
from tqdm import tqdm
import scipy.linalg

In [2]:

active_all_event_ids = {'769': 769, '770': 770, '771': 771, '772': 772}
active_lr_event_ids = {'769': 769, '770': 770}
unknown_event_id = {'783': 783} 
data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCICIV_2a'

In [3]:
def affine_transform(covmats, reference):
    """
    Apply the affine transformation:
    C -> R^(-1/2) * C * R^(-1/2)
    """
    ref_sqrt_inv = np.linalg.inv(scipy.linalg.sqrtm(reference))
    return np.array([ref_sqrt_inv @ c @ ref_sqrt_inv for c in covmats])

In [4]:
def mdm_classify(cov_train, y_train, cov_test, y_test):
    """
    Minimum Distance to Mean (MDM) classification.

    1) Compute Riemannian mean for each class
    2) Assign each test trial to class with min Riemannian distance
    """
    classes = np.unique(y_train)
    means = {}
    for c in classes:
        means[c] = mean_riemann(cov_train[y_train == c])
    
    predictions = []
    for test_cov in cov_test:
        # Compute distance to each class mean
        dists = {}
        for c in classes:
            dists[c] = distance_riemann(test_cov, means[c])
        # Pick class with minimal distance
        predictions.append(min(dists, key=dists.get))
    
    return np.mean(np.array(predictions) == y_test)

In [5]:
def cov(X):
    covs = np.array([trial @ trial.T for trial in X])
    return covs

In [6]:

def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

In [7]:

# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data


In [8]:

# With a sampling frequency of 250 Hz, 1001 samples equate to 1001/250 seconds.
sfreq = 250
tmin = 0.5       # Epoch start at cue onset.
# Set tmax so that n_samples = (tmax-tmin)*sfreq + 1 = 1001, i.e. 4 seconds long.
tmax = 3.5  # This gives 4.0 seconds.
filter_order = 50

In [9]:
preproc_options = {
        'None': None,
        'Standard': (8, 30),
        'Alpha': (8, 12),
        'Beta': (13, 30)
    }

In [10]:

def twofour_crosssession(n_classes):
    for proc_name, freq_range in preproc_options.items():
        if(n_classes==2):
            event_ids = active_lr_event_ids
        else:
            event_ids = active_all_event_ids
        

        train_active_X = []         # List to hold numpy arrays with shape (n_trials, 22, 1001) per subject.
        train_active_y = []         # List to hold event labels per subject.
        train_active_metadata = []  # List to hold event metadata per subject.

        # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
        for subj in range(1, 10):
            filename = os.path.join(data_dir, f'A{subj:02d}T.gdf')
            
            # Read the GDF file (using preload=True to load data into memory).
            train_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
            # Retain only EEG channels (22 channels) and exclude EOG channels.
            train_raw.pick_types(eeg=True, eog=False)
            
            # Extract events corresponding only to the four desired types.
            train_active_events, _ = mne.events_from_annotations(train_raw, event_id=event_ids)
            
            # Create epochs from tmin to tmax.
            train_active_epochs = mne.Epochs(train_raw, train_active_events, event_id=event_ids, tmin=tmin, tmax=tmax,
                                baseline=None, preload=True, verbose=False)
            
            # Get the epoch data (num_epochs x 22 channels x 1001 samples).
            train_active_data = train_active_epochs.get_data()
            
            # Print the number of extracted epochs to verify
            print(f"Subject {subj}: Epoch data shape {train_active_data.shape}")
            
            # Sampling frequency from raw.info (should be 250).
            fs = int(train_raw.info['sfreq'])
            # print(fs)
            n_trials, n_channels, n_times = train_active_data.shape
            train_active_filtered_data = np.empty_like(train_active_data)
            
            # Apply the causal bandpass filter channel‐wise for each trial.
            if freq_range is None:
                train_active_filtered_data = train_active_data.copy()
            else:
                lowcut, highcut = freq_range
                for trial in range(n_trials):
                    for ch in range(n_channels):
                        train_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                            train_active_data[trial, ch, :],
                            lowcut=lowcut,    # Lower bound of sensorimotor rhythm.
                            highcut=highcut,  # Upper bound of sensorimotor rhythm.
                            fs=fs,
                            order=filter_order     # Lower order for a smoother causal filter.
                        )
            # Apply the causal bandpass filter channel‐wise for each trial.
            # for trial in range(n_trials):
            #     for ch in range(n_channels):
            #         train_active_filtered_data[trial, ch, :] = train_active_data[trial, ch, :]
            
            # Append the processed data, labels, and event metadata.
            train_active_X.append(train_active_filtered_data)
            train_active_y.append(train_active_epochs.events[:, 2])  # The third column holds the event code.
            train_active_metadata.append(train_active_epochs.events)

        print("Loaded data for", len(train_active_X), "subjects.")

        eval_active_X = []         
        eval_active_y = []        
        eval_active_metadata = [] 

        # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
        for subj in range(1, 10):
            filename = os.path.join(data_dir, f'A{subj:02d}E.gdf')
            mat_data = loadmat(f'/home/vishwa/eeg_tl/Recreating papers/BCICIV_2A true labels/A{subj:02d}E.mat')
            true_y =  np.array(mat_data['classlabel'], dtype=np.int64).reshape(288,) + 768
            # Read the GDF file (using preload=True to load data into memory).
            eval_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
            # Retain only EEG channels (22 channels) and exclude EOG channels.
            eval_raw.pick_types(eeg=True, eog=False)
            
            # Extract events corresponding only to the four desired types.
            eval_active_events, _ = mne.events_from_annotations(eval_raw, event_id=unknown_event_id)
            
            # Create epochs from tmin to tmax.
            eval_active_epochs = mne.Epochs(eval_raw, eval_active_events, event_id=unknown_event_id, tmin=tmin, tmax=tmax,
                                baseline=None, preload=True, verbose=False)
            
            # Get the epoch data (num_epochs x 22 channels x 1001 samples).
            eval_active_data = eval_active_epochs.get_data()
            
            # Print the number of extracted epochs to verify
            print(f"Subject {subj}: Epoch data shape {eval_active_data.shape}")
            
            # Sampling frequency from raw.info (should be 250).
            fs = int(eval_raw.info['sfreq'])
            # print(fs)
            n_trials, n_channels, n_times = eval_active_data.shape
            eval_active_filtered_data = np.empty_like(eval_active_data)
            
            # Apply the causal bandpass filter channel‐wise for each trial.
            if freq_range is None:
                eval_active_filtered_data = eval_active_data.copy()
            else:
                lowcut, highcut = freq_range
                for trial in range(n_trials):
                    for ch in range(n_channels):
                        eval_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                            eval_active_data[trial, ch, :],
                            lowcut=lowcut,    # Lower bound of sensorimotor rhythm.
                            highcut=highcut,  # Upper bound of sensorimotor rhythm.
                            fs=fs,
                            order=filter_order     # Lower order for a smoother causal filter.
                        )
            
                    
            
            # Append the processed data, labels, and event metadata.
            eval_active_X.append(eval_active_filtered_data)
            eval_active_y.append(true_y)  # The third column holds the event code.
            eval_active_metadata.append(eval_active_epochs.events)

        print("Loaded data for", len(eval_active_X), "subjects.")

        train_resting_X = []
        train_resting_metadata = []

        for subj in range(1, 10):
            filename = os.path.join(data_dir, f'A{subj:02d}T.gdf')
            
            # Load the GDF file and select EEG channels.
            train_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, 
                                    eog=['EOG-left', 'EOG-central', 'EOG-right'])
            train_raw.pick_types(eeg=True, eog=False)
            
            # Extract active events (e.g. left/right motor imagery cues).
            train_resting_events, _ = mne.events_from_annotations(train_raw, event_id=active_all_event_ids)
            
            # Create active epochs.
            train_resting_epochs = mne.Epochs(train_raw, train_resting_events, event_id=active_all_event_ids,
                                    tmin=tmin, tmax=tmax, baseline=None, preload=True, verbose=False)
            
            # Now derive resting events by shifting each active event by 6 seconds (trial end).
            fs = int(train_raw.info['sfreq'])
            train_resting_events[:, 0] += int(6 * fs)
            
            # Create resting epochs of 1.5 s duration.
            train_resting_epochs = mne.Epochs(train_raw, train_resting_events, tmin=0, tmax=1.5,
                                        baseline=None, preload=True, verbose=False)
            
            train_resting_data = train_resting_epochs.get_data()
            print(f"Subject {subj}: Resting data shape {train_resting_data.shape}")
            
            n_trials, n_channels, n_times = train_resting_data.shape
            train_resting_filtered_data = np.empty_like(train_resting_data)
            
            if freq_range is None:
                train_resting_filtered_data = train_resting_data.copy()
            else:
                lowcut, highcut = freq_range
                for trial in range(n_trials):
                    for ch in range(n_channels):
                        train_resting_filtered_data[trial, ch, :] = causal_bandpass_filter(
                            train_resting_data[trial, ch, :],
                            lowcut=lowcut,    # Lower bound of sensorimotor rhythm.
                            highcut=highcut,  # Upper bound of sensorimotor rhythm.
                            fs=fs,
                            order=filter_order     # Lower order for a smoother causal filter.
                        )
            
            train_resting_X.append(train_resting_filtered_data)
            train_resting_metadata.append(train_resting_epochs.events)

        print("Loaded resting data for", len(train_resting_X), "subjects.")

        eval_resting_X = []
        eval_resting_metadata = []

        for subj in range(1, 10):
            filename = os.path.join(data_dir, f'A{subj:02d}E.gdf')
        
            # Load the GDF file and select EEG channels.
            eval_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False,
                                    eog=['EOG-left', 'EOG-central', 'EOG-right'])
            eval_raw.pick_types(eeg=True, eog=False)
        
            # Extract resting events (e.g. left/right motor imagery cues).
            eval_resting_events, _ = mne.events_from_annotations(eval_raw, event_id=unknown_event_id)
        
            # Create resting epochs.
            eval_resting_epochs = mne.Epochs(eval_raw, eval_resting_events, event_id=unknown_event_id,
                                    tmin=tmin, tmax=tmax, baseline=None, preload=True, verbose=False)
        
            # Now derive resting events by shifting each resting event by 6 seconds (trial end).
            fs = int(eval_raw.info['sfreq'])
            # eval_resting_events = eval_resting_events.copy()
            eval_resting_events[:, 0] += int(6 * fs)
        
            # Create resting epochs of 1.5 s duration.
            eval_resting_epochs = mne.Epochs(eval_raw, eval_resting_events, tmin=0, tmax=1.5,
                                        baseline=None, preload=True, verbose=False)
        
            eval_resting_data = eval_resting_epochs.get_data()
            print(f"Subject {subj}: Resting data shape {eval_resting_data.shape}")
        
            n_trials, n_channels, n_times = eval_resting_data.shape
            eval_resting_filtered_data = np.empty_like(eval_resting_data)
        
            if freq_range is None:
                eval_resting_filtered_data = eval_resting_data.copy()
            else:
                lowcut, highcut = freq_range
                for trial in range(n_trials):
                    for ch in range(n_channels):
                        eval_resting_filtered_data[trial, ch, :] = causal_bandpass_filter(
                            eval_resting_data[trial, ch, :],
                            lowcut=lowcut,    # Lower bound of sensorimotor rhythm.
                            highcut=highcut,  # Upper bound of sensorimotor rhythm.
                            fs=fs,
                            order=filter_order     # Lower order for a smoother causal filter.
                        )
        
            eval_resting_X.append(eval_resting_filtered_data)
            # For resting epochs, labels might not be needed or can be set to a dummy value.
            eval_resting_metadata.append(eval_resting_epochs.events)


        print("Loaded resting data for", len(eval_resting_X), "subjects.")

        if(n_classes==2):
            eval_active_X = [x[np.isin(y, [769, 770])] for x, y in zip(eval_active_X, eval_active_y)]
            eval_active_y = [y[np.isin(y, [769, 770])] for y in eval_active_y]
        
        train_active_y = encode_labels(train_active_y)
        eval_active_y = encode_labels(eval_active_y)

        train_active_covs = [cov(i) for i in train_active_X]
        eval_active_covs = [cov(i) for i in eval_active_X]

        train_resting_covs = [cov(i) for i in train_resting_X]
        eval_resting_covs =  [cov(i) for i in eval_resting_X]

        train_resting_covs_mean = [mean_riemann(i) for i in train_resting_covs]
        eval_resting_covs_mean = [mean_riemann(i) for i in eval_resting_covs]
        # train_resting_covs = [cov(i) for i in train_active_X]
        # eval_resting_covs =  [cov(i) for i in eval_active_X]

        affine_transformed_train_covs = [affine_transform(train_active_covs[i], train_resting_covs_mean[i]) for i in range(len(train_active_covs))]
        affine_transformed_eval_covs = [affine_transform(eval_active_covs[i], eval_resting_covs_mean[i]) for i in range(len(eval_active_covs))]

        accuracies = []

        for subj_idx in range(len(train_active_X)):
            # print(subj_idx)
            # Split data into train/test using leave-one-subject-out
            X_test = affine_transformed_eval_covs[subj_idx]
            y_test = eval_active_y[subj_idx]
            
            # Concatenate data from other subjects
            X_train = affine_transformed_train_covs[subj_idx]
            y_train = train_active_y[subj_idx]

            accuracy = mdm_classify(X_train, y_train, X_test, y_test)
            accuracies.append(accuracy)
        print(proc_name)
        for subj_idx in range(len(train_active_X)):
            print(f"Subject {subj_idx+1} Test Accuracy: {accuracies[subj_idx]:.2f}")

        print(f"\nMean Cross-Validation Accuracy: {np.mean(accuracies):.2f} ± {np.std(accuracies):.2f}")
        

In [11]:
twofour_crosssession(4)

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 1: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 2: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 7: Resting data shape (287, 22, 376)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 8: Resting data shape (287, 22, 376)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 9: Resting data shape (287, 22, 376)
Loaded resting data for 9 subjects.
Beta
Subject 1 Test Accuracy: 0.83
Subject 2 Test Accuracy: 0.56
Subject 3 Test Accuracy: 0.76
Subject 4 Test Accuracy: 0.67
Subject 5 Test Accuracy: 0.46
Subject 6 Test Accuracy: 0.51
Subject 7 Test Accuracy: 0.84
Subject 8 Test Accuracy: 0.69
Subject 9 Test Accuracy: 0.72

Mean Cross-Validation Accuracy: 0.67 ± 0.13
